In [ ]:
import pandas as pd
from google.colab import files
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

In [ ]:
# Setup Chrome Driver
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0")
service = Service('/usr/bin/chromedriver')
driver = webdriver.Chrome(options=options)

# Data containers
year_brand, model, total_price, price_per_month = [], [], [], []
mileage, transmission, location, highlight, links = [], [], [], [], []

# Loop page
for page in range(1, 103):
    url = f"https://www.carsome.my/buy-car?page=%7Bpage%7D&pageNo={page}"
    driver.get(url)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "mod-b-card"))
        )
    except:
        print(f"❌ Gagal load data mobil di page {page}")
        continue

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    car_data = soup.find_all('article', class_='mod-b-card')
    # print(car_data[0].petrify())

    print(f"📄 Page {page}: {len(car_data)} cars found")
    if len(car_data) == 0:
        break

    for store in car_data:
        try:
            # Title: Year + Brand + Model
            title_tag = store.find('a', class_='mod-b-card__title')
            if title_tag:
                full_text = title_tag.get_text(strip=True)
                parts = full_text.split()
                year = parts[0]
                brand_model = " ".join(parts[1:])
                link = "https://www.carsome.my" + title_tag['href']
            else:
                year, brand_model, link = "-", "-", "-"

            # Total Price
            price_total = store.find('div', class_='mod-card__price')
            price = price_total.find_all('span')[-1].text.strip() if price_total else "-"

            # Price per Month (ambil hanya angka RM-nya saja)
            ppm_tag = store.find('div', class_='mod-tooltipMonthPay')
            ppm_raw = ppm_tag.get_text(strip=True) if ppm_tag else "-"
            ppm = ppm_raw.split("/")[0].strip() if "/" in ppm_raw else ppm_raw

            # Mileage & Transmission
            car_other = store.find('div', class_='mod-b-card__car-other')
            spans = car_other.find_all('span') if car_other else []
            km = spans[0].text.strip() if len(spans) > 0 else "-"
            trans = spans[1].text.strip() if len(spans) > 1 else "-"

            # Location
            loc_tag = store.find('div', class_='mod-b-card__car-location')
            loc = loc_tag.text.strip() if loc_tag else "-"

            # Highlight (Family Drive, Daily Drive, dll)
            tag = store.find('div', class_='mod-car-tagging')
            span_tag = tag.find('span') if tag else None
            hl = span_tag.get_text(strip=True) if span_tag else "-"

            # Append ke list
            year_brand.append(year)
            model.append(brand_model)
            total_price.append(price)
            price_per_month.append(ppm)
            mileage.append(km)
            transmission.append(trans)
            location.append(loc)
            highlight.append(hl)
            links.append(link)

        except Exception as e:
            print(f"⚠️ Gagal ambil 1 mobil: {e}")
            continue

    time.sleep(2)  # delay antar halaman

# close browser
driver.quit()

# DataFrame
df = pd.DataFrame({
    'Year': year_brand,
    'Model': model,
    'Total_Price': total_price,
    'Price_per_month': price_per_month,
    'Mileage': mileage,
    'Transmission': transmission,
    'Location': location,
    'Highlight': highlight,
    'URL': links
})

df.head()

📄 Page 1: 18 cars found
📄 Page 2: 18 cars found
📄 Page 3: 18 cars found
📄 Page 4: 18 cars found
📄 Page 5: 18 cars found
📄 Page 6: 18 cars found
📄 Page 7: 18 cars found
📄 Page 8: 18 cars found
📄 Page 9: 18 cars found
📄 Page 10: 18 cars found
📄 Page 11: 18 cars found
📄 Page 12: 18 cars found
📄 Page 13: 18 cars found
📄 Page 14: 18 cars found
📄 Page 15: 18 cars found
📄 Page 16: 18 cars found
📄 Page 17: 18 cars found
📄 Page 18: 18 cars found
📄 Page 19: 18 cars found
📄 Page 20: 18 cars found
📄 Page 21: 18 cars found
📄 Page 22: 18 cars found
📄 Page 23: 18 cars found
📄 Page 24: 18 cars found
📄 Page 25: 18 cars found
📄 Page 26: 18 cars found
📄 Page 27: 18 cars found
📄 Page 28: 18 cars found
📄 Page 29: 18 cars found
📄 Page 30: 18 cars found
📄 Page 31: 18 cars found
📄 Page 32: 18 cars found
📄 Page 33: 18 cars found
📄 Page 34: 18 cars found
📄 Page 35: 18 cars found
📄 Page 36: 18 cars found
📄 Page 37: 18 cars found
📄 Page 38: 18 cars found
📄 Page 39: 18 cars found
📄 Page 40: 18 cars found
📄 Page 41

,Year,Model,Total_Price,Price_per_month,Mileage,Transmission,Location,Highlight,URL
0,2017,Perodua Myvi SE 1.5,"RM\n 35,400",RM 406,"92,189 km",Automatic,"CARSOME Showroom Setia SPICE, Pulau Pinang 14,...",National Daily Drive,https://www.carsome.my/buy-car/perodua/myvi/20...
1,2023,Subaru Forester L EyeSight 2.0,"RM\n 115,800","RM 1,137","14,200 km",Automatic,"CARSOME PJ Automall, Selangor 14,748.1 km\n ...",Compact SUV,https://www.carsome.my/buy-car/subaru/forester...
2,2018,Honda Jazz S i-VTEC 1.5,"RM\n 52,900",RM 611,"75,759 km",Automatic,"CARSOME Showroom Kuala Terengganu, Terengganu ...",Daily Drive,https://www.carsome.my/buy-car/honda/jazz/2018...
3,2016,Mazda CX-3 SKYACTIV 2.0,"RM\n 55,800",RM 711,"59,833 km",Automatic,"CARSOME Showroom Kota Kinabalu, Sabah 13,814.5...",Compact SUV,https://www.carsome.my/buy-car/mazda/cx-3/2016...
4,2021,Perodua Myvi AV 1.5,"RM\n 49,900",RM 460,"69,974 km",Automatic,"CARSOME Showroom Johor Jaya, Johor 14,832.8 km...",National Daily Drive,https://www.carsome.my/buy-car/perodua/myvi/20...


In [ ]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.to_csv('carsome_data_raw.csv', index=False)
files.download('carsome_data_raw.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>